# Lab 3 — Workflow Multi-Agent Pattern (Developer-Defined)

**Scenario:** Investigate synthetic incident **INC-1042 (checkout-api latency)** using a repeatable workflow with **parallel** Metrics and Logs Agents followed by a Report Writer Agent.

**Pattern:** `Collect metadata → [Metrics Agent || Logs Agent] → Report Writer → Save checkpoint`

**Learning objectives**
1. Create specialized Strands `Agent` instances and give each a read-only tool.
2. Define task order/dependencies with Python.
3. Run independent Agents concurrently, then wait for **both** results before reporting.
4. Use Memory, automatic Context Management and Session Management with clear scope.
5. Capture execution duration, outputs and a workflow-level checkpoint.




## Step 1 — Locate the project and check imports

This cell works when Jupyter's working directory is either the extracted project root or its immediate `notebooks/` folder. If the assertion fails, open Jupyter from the extracted lab project.

In [1]:
import json
import os
import sys
import time
import uuid
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "common.py").is_file(), f"Cannot find common.py at {ROOT}; launch Jupyter from the extracted project."
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from strands.session import SnapshotSessionManager
from strands.storage import LocalFileStorage
from common import INCIDENT, STATE, build_agent, get_incident_metrics, get_incident_logs, MEMORY_FILE

print("Project root:", ROOT)
print("Incident:", INCIDENT["incident_id"])
print("State folder:", STATE)
print("Memory store path:", MEMORY_FILE)

Project root: C:\Users\LeCaoThiHoang\strands_multiagent_labs
Incident: INC-1042
State folder: C:\Users\LeCaoThiHoang\strands_multiagent_labs\state
Memory store path: C:\Users\LeCaoThiHoang\strands_multiagent_labs\state\operator_preferences.json


## Step 2 — Inspect synthetic evidence (no model calls)

These are local training fixtures. Inspecting them does not invoke Bedrock or another LLM.

In [2]:
print(json.dumps({
    "incident_id": INCIDENT["incident_id"],
    "service": INCIDENT["service"],
    "p95_latency_ms": INCIDENT["p95_latency_ms"],
    "http_5xx_pct": INCIDENT["http_5xx_pct"],
    "database_connections": INCIDENT["database_connections"],
    "recent_change": INCIDENT["recent_change"],
}, indent=2))
assert INCIDENT["incident_id"] == "INC-1042"

{
  "incident_id": "INC-1042",
  "service": "checkout-api",
  "p95_latency_ms": {
    "baseline": 180,
    "observed": 2450
  },
  "http_5xx_pct": 8.1,
  "database_connections": {
    "max": 100,
    "observed": 99
  },
  "recent_change": "checkout-api release 2.7 at 07:55 UTC"
}


## Step 3 — Define three specialized Strands Agents

- **Metrics Agent:** calls `get_incident_metrics`, returns metric evidence and uncertainty.
- **Logs Agent:** calls `get_incident_logs`, returns log evidence and uncertainty.
- **Report Writer:** receives both analyses; writes **Evidence / Hypothesis / Recommended Checks / Uncertainty**.

The shared `build_agent()` helper configures `context_manager="auto"` for all Agents. The Report Writer also receives a `MemoryManager` configured for retrieving saved report preferences (when available).

A unique session ID is used for each notebook run to avoid unintentionally mixing previous report history into the new investigation. To deliberately test session continuity, reuse the same session ID in a separate run.

In [3]:
RUN_ID = uuid.uuid4().hex[:8]
SESSION_ID = f"workflow-report-inc1042-{RUN_ID}"

metrics_agent = build_agent(
    name="metrics_analyst",
    description="Analyzes synthetic incident metrics",
    system_prompt=(
        "You are a metrics specialist. For INC-1042, call get_incident_metrics. "
        "Report only observed values and clearly label any hypothesis. "
        "Do not invent data or perform changes."
    ),
    tools=[get_incident_metrics],
)

logs_agent = build_agent(
    name="logs_analyst",
    description="Analyzes synthetic incident log excerpts",
    system_prompt=(
        "You are a log specialist. For INC-1042, call get_incident_logs. "
        "Quote only relevant synthetic evidence and clearly label hypotheses. "
        "Do not invent data or perform changes."
    ),
    tools=[get_incident_logs],
)

report_session = SnapshotSessionManager(
    session_id=SESSION_ID,
    storage=LocalFileStorage(str(STATE / "sessions")),
)

report_agent = build_agent(
    name="incident_report_writer",
    description="Merges metrics and logs findings into an incident report",
    system_prompt=(
        "Write a concise read-only incident report. "
        "Use these headings: Evidence, Hypothesis, Recommended Checks, Uncertainty. "
        "Use only supplied incident evidence; distinguish verified observations from hypotheses. "
        "Never claim that a production remediation has been performed."
    ),
    use_memory=True,
    session_manager=report_session,
)

print("Strands Agents ready")
print("Report Writer session:", SESSION_ID)

Strands Agents ready
Report Writer session: workflow-report-inc1042-a2cc4e52


## Step 4 — Stage 1: Collect incident metadata (deterministic Python)

This task is defined by the developer and needs **no LLM call**.

In [4]:
metadata = {
    "incident_id": INCIDENT["incident_id"],
    "service": INCIDENT["service"],
    "environment": INCIDENT["environment"],
}
print(json.dumps(metadata, indent=2))

{
  "incident_id": "INC-1042",
  "service": "checkout-api",
  "environment": "training-sandbox"
}


## Step 5 — Stage 2: Run Metrics and Logs Agents in parallel

This is the key **Workflow dependency**: both independent analyses must complete before the Report Writer starts. Each worker uses a **different Agent instance**, avoiding concurrent invocations of the same mutable Agent instance.

These are real LLM invocations. Watch the tool calls in the notebook output; the exact wording of the results can vary.

In [5]:
parallel_start = time.monotonic()

with ThreadPoolExecutor(max_workers=2) as pool:
    metrics_future = pool.submit(
        metrics_agent,
        "Investigate synthetic incident INC-1042. Call your read-only metrics tool and report evidence."
    )
    logs_future = pool.submit(
        logs_agent,
        "Investigate synthetic incident INC-1042. Call your read-only logs tool and report evidence."
    )

    # Both future.result() calls must return successfully before Stage 3 starts.
    metrics_response = metrics_future.result()
    logs_response = logs_future.result()

metrics_result = str(metrics_response)
logs_result = str(logs_response)
parallel_seconds = round(time.monotonic() - parallel_start, 2)

assert metrics_result.strip(), "Metrics Agent returned an empty result"
assert logs_result.strip(), "Logs Agent returned an empty result"
print("Parallel analysis completed in", parallel_seconds, "seconds")
print("\nMetrics findings (excerpt):\n", metrics_result[:1200])
print("\nLogs findings (excerpt):\n", logs_result[:1200])

SureSure! Let me fetch the synthetic metrics for INC-1042 right! Let me fetch the synthetic incident logs for INC-1042 right away. away.
Tool #1: get_incident_metrics

Tool #1: get_incident_logs
Here isHere the full evidence report for **INC-1042**, based solely on the synthetic read-only metrics returned is the full evidence-based report for **INC-1042**, using only by the tool. No changes were made, and no real AWS data was accessed.

---

## 🔍 INC-1042 — Synthetic Metrics Report

| the synthetic log data returned by the tool:

---

## 🔍 Incident INC-1042 — Investigation Report

### 📋 Raw Synthetic Evidence (Quoted Directly from Field | Value |
|---|---|
| **Service** | `checkout-api` |
| **Metric Window** | 2026-11-10 08:00  Logs)

| Timestamp | Level | Message |
|-----------|-------|---------|
| `08:03:10` | **ERROR** | `checkout-api databaseUTC → 08:15 UTC (15 min) |

---

### 📊 Observed connection acquisition timed out` |
| `08:04:11` | **WARN** | `checkout-api connection pool ne

**Validation checkpoint:** The two `future.result()` calls implement the join. If either Agent raises an exception, execution stops before the Report Writer. Independent Agents may finish in either order; the Report Writer waits for both.

**Reminder:** The Agents' answers are not guaranteed to be factually correct. Validate their claims against the synthetic fixture and tool output.

## Step 6 — Stage 3: Combine outputs into one report

Here the developer passes **both upstream results** explicitly to the Report Writer. That is output propagation in a developer-defined Workflow.

The Report Writer has Session Management and read-capable Memory configured. It may retrieve a report-format preference saved in Lab 1 if the memory exists and the model chooses to retrieve it. This is **not** a proof that a memory lookup occurred: inspect the tool trace or saved memory separately.

In [6]:
report_input = {
    "incident_metadata": metadata,
    "metrics_findings": metrics_result,
    "logs_findings": logs_result,
}
report_prompt = (
    "Create an incident report for INC-1042 from this evidence only. "
    "Use Evidence, Hypothesis, Recommended Checks, Uncertainty as headings. "
    "Do not perform remediation.\n\n"
    + json.dumps(report_input, indent=2)
)

report_start = time.monotonic()
report_response = report_agent(report_prompt)
report_text = str(report_response)
report_seconds = round(time.monotonic() - report_start, 2)

assert report_text.strip(), "Report Writer returned an empty report"
print("Report Agent duration:", report_seconds, "seconds")
print("\nFINAL INCIDENT REPORT:\n")
print(report_text)


Tool #1: search_memory
---

# Incident Report — INC-1042

**Service:** `checkout-api` | **Environment:** `training-sandbox` | **Window:** 2026-11-10 08:00–08:15 UTC
*All findings are derived exclusively from synthetic lab evidence. No real AWS checks were performed and no production changes were made.*

---

## Evidence

**Verified observations from supplied synthetic data only:**

| # | Source | Observation |
|---|--------|-------------|
| 1 | Metrics | P95 latency: **2,450 ms** vs. 180 ms baseline (+1,261%; ~13.6× above normal) |
| 2 | Metrics | HTTP 5xx error rate: **8.1%** during the window |
| 3 | Metrics | DB connection pool utilisation: **99/100 connections in use** |
| 4 | Metrics | CPU utilisation: **48%** — within moderate, non-critical range |
| 5 | Logs | `08:03:10` ERROR — `checkout-api database connection acquisition timed out` |
| 6 | Logs | `08:04:11` WARN — `checkout-api connection pool near capacity` |
| 7 | Logs | `08:06:05` ERROR — `checkout-api request failed: DB 

## Step 7 — Save a workflow-level checkpoint

The Report Writer's `SnapshotSessionManager` persists its **agent conversation state**. This separate JSON checkpoint saves a small **application-level summary** of the developer-defined Workflow. It does **not** automatically restore a partially completed `ThreadPoolExecutor` DAG.

In [ ]:
checkpoint = {
    "run_id": RUN_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "incident_id": metadata["incident_id"],
    "pattern": "developer_defined_workflow",
    "steps": ["collect_metadata", "parallel_metrics_and_logs", "write_report"],
    "parallel_analysis_seconds": parallel_seconds,
    "report_seconds": report_seconds,
    "report_session_id": SESSION_ID,
    "report_excerpt": report_text[:1200],
}
checkpoint_path = STATE / f"workflow_checkpoint_{RUN_ID}.json"
checkpoint_path.write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")
print("Checkpoint:", checkpoint_path)
print("Saved:", checkpoint_path.exists())
print(json.dumps(json.loads(checkpoint_path.read_text(encoding="utf-8")), indent=2)[:1500])

## Step 8 — Validate lab outcomes

Run this cell to check the observable artifacts. Note that no check below **proves** the model used the memory store or that ContextManager actually compressed a message; those require reviewing tool calls / context telemetry under a suitable test scenario.

In [ ]:
checks = {
    "Synthetic incident loaded": metadata["incident_id"] == "INC-1042",
    "Metrics analysis returned content": bool(metrics_result.strip()),
    "Logs analysis returned content": bool(logs_result.strip()),
    "Final report returned content": bool(report_text.strip()),
    "Workflow checkpoint saved": checkpoint_path.exists(),
    "Report Writer session configured": bool(SESSION_ID),
}
for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "—", name)
assert all(checks.values())

## Step 9 — Exercises

1. **Dependencies:** Remove the parallel stage and call the two Agents sequentially. Compare latency. Parallel execution may improve end-to-end time, but model/service concurrency and overhead affect the result.
2. **Fault handling:** Temporarily make one worker raise an exception and observe that report generation does not proceed. Add an explicit retry policy only if actions are safe to retry.
3. **Memory:** Run Lab 1 to save a report preference; then inspect the Report Writer's tool calls to check whether `search_memory` was invoked. Do not assume memory retrieval from the final report alone.
4. **Session:** Reuse a deliberately chosen session ID in a new run and examine whether the Report Writer restores its conversation context. Keep incident runs isolated by default.
5. **Observability:** Enable `LAB_TRACE=1` *before imports / agent creation* in a fresh kernel and call `setup_observability()` from `common.py`. Inspect model/tool spans. Treat prompts/logs as sensitive telemetry.
6. **Context Management:** The helper configures `context_manager="auto"`; short synthetic data may not trigger reduction. Test with longer safe synthetic inputs and verify behavior through tracing.

### Expected result
A report with Evidence, Hypothesis, Recommended Checks and Uncertainty; two independent analysis Agents; a saved workflow checkpoint; and Report Writer session storage.

### Differences from Lab 2 — Swarm
- **Swarm:** Strands' native orchestrator; Agents autonomously hand off work.
- **Workflow (this lab):** Developer defines dependencies and the join; Agents do not choose the next step.

**Documentation:** [Strands Workflow](https://strandsagents.com/docs/user-guide/sdk/multi-agent/workflow/) · [Strands Session Management](https://strandsagents.com/docs/user-guide/sdk/agents/session-management/)